In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import Matern, RationalQuadratic, RBF
from src.models.mlp import create_mlp_pytorch
from src.models.resnet import ResBlock, ResNet
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterGrid
import shap

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No no

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import Tuple, List, Optional
from copy import deepcopy

class EarlyStopping:
    """Early stopping to stop training when validation loss doesn't improve."""
    def __init__(self, patience: int = 10, min_delta: float = 0.0, verbose: bool = False):
        """
        Args:
            patience: Number of epochs to wait after last improvement
            min_delta: Minimum change to qualify as improvement
            verbose: If True, prints early stopping messages
        """
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_loss: float, model: nn.Module) -> bool:
        """
        Check if training should stop early.
        
        Returns:
            True if training should stop, False otherwise
        """
        if val_loss < self.best_loss - self.min_delta:
            if self.verbose:
                print(f'Validation loss decreased ({self.best_loss:.6f} -> {val_loss:.6f}). Saving model...')
            self.best_loss = val_loss
            self.counter = 0
            # Save the best model state
            self.best_model_state = deepcopy(model.state_dict())
            return False
        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print("Early stopping triggered")
            return self.early_stop

In [3]:
descriptor_df = pd.read_csv('data/bace/external_descriptors.csv')
smiles_df = pd.read_csv('data/bace/smiles.csv')
learned_descriptors_df = pd.read_csv('data/bace/learned_predictors_0.csv')



In [4]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

class ScaffoldSplitter:
    """Comprehensive scaffold splitter with train/val/test splits."""
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        np.random.seed(random_state)
    
    def get_scaffold(self, smiles):
        """Extract scaffold from SMILES."""
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return "invalid"
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            return Chem.MolToSmiles(scaffold)
        except:
            return "invalid"
    
    def split(self, smiles_list, y=None, 
              train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
        """
        Split into train/validation/test sets by scaffold.
        
        Args:
            smiles_list: List of SMILES strings
            y: Target values (optional)
            train_ratio, val_ratio, test_ratio: Split ratios
            
        Returns:
            Indices for each split
        """
        # Generate scaffolds
        scaffolds = [self.get_scaffold(s) for s in smiles_list]
        
        # Create groups dictionary
        scaffold_to_indices = {}
        for idx, scaffold in enumerate(scaffolds):
            if scaffold not in scaffold_to_indices:
                scaffold_to_indices[scaffold] = []
            scaffold_to_indices[scaffold].append(idx)
        
        # Sort scaffolds by number of molecules (descending)
        sorted_scaffolds = sorted(scaffold_to_indices.items(),
                                 key=lambda x: len(x[1]), reverse=True)
        
        # Initialize split indices
        train_idx, val_idx, test_idx = [], [], []
        
        # Distribute scaffolds to splits
        for scaffold, indices in sorted_scaffolds:
            if len(train_idx) / len(smiles_list) < train_ratio:
                train_idx.extend(indices)
            elif len(val_idx) / len(smiles_list) < val_ratio:
                val_idx.extend(indices)
            else:
                test_idx.extend(indices)
        
        # Shuffle indices
        np.random.shuffle(train_idx)
        np.random.shuffle(val_idx)
        np.random.shuffle(test_idx)
        
        return train_idx, val_idx, test_idx


In [5]:
# Create splitter
splitter = ScaffoldSplitter(random_state=42)

# Get split indices
train_idx, val_idx, test_idx = splitter.split(
    smiles_df['ids'].tolist(),
    y=smiles_df['y'].tolist(),
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2)


In [6]:
smiles_df_train = smiles_df.iloc[train_idx].reset_index(drop=True)
smiles_df_val = smiles_df.iloc[val_idx].reset_index(drop=True)
smiles_df_test = smiles_df.iloc[test_idx].reset_index(drop=True)
    

In [7]:


descriptor_df_train = descriptor_df.iloc[train_idx].reset_index(drop=True)
descriptor_df_val = descriptor_df.iloc[val_idx].reset_index(drop=True)
descriptor_df_test = descriptor_df.iloc[test_idx].reset_index(drop=True)
    

In [8]:
learned_descriptors_df_train = learned_descriptors_df.iloc[train_idx].reset_index()
learned_descriptors_df_val = learned_descriptors_df.iloc[val_idx].reset_index()
learned_descriptors_df_test = learned_descriptors_df.iloc[test_idx].reset_index()
    

In [9]:
df_train = pd.concat([learned_descriptors_df_train, descriptor_df_train, smiles_df_train], axis=1)
df_val = pd.concat([learned_descriptors_df_val, descriptor_df_val, smiles_df_val], axis=1)
df_test = pd.concat([learned_descriptors_df_test, descriptor_df_test, smiles_df_test],axis=1)

In [10]:
df_train.head()

,index,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,...,209,210,211,212,213,214,215,y,w,ids
0,1011,0.015950,0.0,0.132767,0.047984,0.044219,0.000000,0.048745,0.011339,0.000234,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.844512,O=C1N(C)C(=N[C@@](C1)(CCc1ccccc1)C)N
1,490,0.048197,0.0,0.204776,0.072178,0.073979,0.000125,0.032347,0.004190,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.844512,n1ccc(cc1)C1(N=C(N)c2c1cccc2)c1cc(ccc1)-c1cncnc1
2,1210,0.028600,0.0,0.238802,0.090424,0.082761,0.000000,0.090894,0.025848,0.000116,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.844512,S1(=O)CC(Cc2cc(O)c(N)c(F)c2)C(O)C([NH2+]Cc2cc(...
3,717,0.056243,0.0,0.239946,0.084071,0.087759,0.000010,0.037141,0.006602,0.000567,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.184116,Clc1ccccc1-c1n(Cc2nc(N)ccc2)c(cc1)-c1ccc(Oc2cn...
4,944,0.038900,0.0,0.210920,0.079502,0.077362,0.000226,0.053684,0.016053,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.844512,Fc1ncccc1-c1cc(ccc1)C1(N=C(OC1)N)c1ccc(OC(F)F)cc1


In [11]:
X_train = df_train.drop(columns=['y', 'w', 'ids'])
y_train = df_train['y']
X_val = df_val.drop(columns=['y', 'ids', 'w'])
y_val = df_val['y']
X_test = df_test.drop(columns=['y', 'ids', 'w'])
y_test = df_test['y']


In [12]:
base_seed = 42
config_id = 0
models_list = []

In [13]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KNeighborsClassifier()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'roc': roc
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 0, 'model_type': 'KNeighborsClassifier', 'hyperparams': {}, 'roc': 0.7673527037933817}
{'index': 1, 'model_type': 'KNeighborsClassifier', 'hyperparams': {}, 'roc': 0.7045332257196664}
{'index': 2, 'model_type': 'KNeighborsClassifier', 'hyperparams': {}, 'roc': 0.7554479418886199}
{'index': 3, 'model_type': 'KNeighborsClassifier', 'hyperparams': {}, 'roc': 0.7288808178638687}


In [14]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = LinearDiscriminantAnalysis()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'roc': roc
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 4, 'model_type': 'LinearDiscriminantAnalysis', 'hyperparams': {}, 'roc': 0.5936911487758946}
{'index': 5, 'model_type': 'LinearDiscriminantAnalysis', 'hyperparams': {}, 'roc': 0.5268361581920904}
{'index': 6, 'model_type': 'LinearDiscriminantAnalysis', 'hyperparams': {}, 'roc': 0.5813828356201237}
{'index': 7, 'model_type': 'LinearDiscriminantAnalysis', 'hyperparams': {}, 'roc': 0.5607344632768362}


In [15]:
param_grid = {
    'reg_param': [0.1, 0.2, 0.5, 1]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = QuadraticDiscriminantAnalysis(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'roc': roc
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 8, 'model_type': 'QuadraticDiscriminantAnalysis', 'hyperparams': {'reg_param': 0.1}, 'roc': 0.6997578692493946}
{'index': 9, 'model_type': 'QuadraticDiscriminantAnalysis', 'hyperparams': {'reg_param': 0.2}, 'roc': 0.6098331988162498}
{'index': 10, 'model_type': 'QuadraticDiscriminantAnalysis', 'hyperparams': {'reg_param': 0.5}, 'roc': 0.709712133440947}
{'index': 11, 'model_type': 'QuadraticDiscriminantAnalysis', 'hyperparams': {'reg_param': 1}, 'roc': 0.6486413774549368}


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full ran

In [16]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = LogisticRegression()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'roc': roc
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 12, 'model_type': 'LogisticRegression', 'hyperparams': {}, 'roc': 0.730562281409739}
{'index': 13, 'model_type': 'LogisticRegression', 'hyperparams': {}, 'roc': 0.7106537530266344}
{'index': 14, 'model_type': 'LogisticRegression', 'hyperparams': {}, 'roc': 0.6602771051923594}
{'index': 15, 'model_type': 'LogisticRegression', 'hyperparams': {}, 'roc': 0.716908797417272}


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-lear

In [17]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = GaussianNB()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'roc': roc
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 16, 'model_type': 'GaussianNB', 'hyperparams': {}, 'roc': 0.6613532418617165}
{'index': 17, 'model_type': 'GaussianNB', 'hyperparams': {}, 'roc': 0.6177024482109228}
{'index': 18, 'model_type': 'GaussianNB', 'hyperparams': {}, 'roc': 0.6698278181329029}
{'index': 19, 'model_type': 'GaussianNB', 'hyperparams': {}, 'roc': 0.5901937046004843}


In [18]:
import time

In [19]:
hyperparams_list = [
    {'max_depth': 5, 'n_estimators': 10, 'learning_rate': 0.01},
    {'max_depth': 10, 'n_estimators': 10, 'learning_rate': 0.01},
    {'max_depth': 10, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 100, 'learning_rate': 0.01},
    {'max_depth': 3, 'n_estimators': 500, 'learning_rate': 0.1},
    {'max_depth': 3, 'n_estimators': 500, 'learning_rate': 0.2},
    {'max_depth': 3, 'n_estimators': 1000, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 2000, 'learning_rate': 0.2}
]

for idx, params in enumerate(hyperparams_list):

    seed = base_seed + config_id

    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    start_time = time.time()
    model = XGBClassifier(
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        n_jobs=-1,
    )

    model.fit(X_boot_scaled, y_boot)

    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    
    roc = float(roc_auc_score(y_val, y_val_pred))
    training_time = time.time() - start_time
    
    print(f'Training time: {training_time}')
    
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'roc': roc
    }
    
    config_id += 1
    
    print(model_metadata)
    
    models_list.append(model_metadata)

Training time: 0.08070802688598633
{'index': 20, 'model_type': 'XGBClassifier', 'hyperparams': {'max_depth': 5, 'n_estimators': 10, 'learning_rate': 0.01}, 'roc': 0.6503228410008071}
Training time: 0.12401723861694336
{'index': 21, 'model_type': 'XGBClassifier', 'hyperparams': {'max_depth': 10, 'n_estimators': 10, 'learning_rate': 0.01}, 'roc': 0.6264460586494485}
Training time: 0.5544490814208984
{'index': 22, 'model_type': 'XGBClassifier', 'hyperparams': {'max_depth': 10, 'n_estimators': 100, 'learning_rate': 0.1}, 'roc': 0.7565240785579769}
Training time: 0.772442102432251
{'index': 23, 'model_type': 'XGBClassifier', 'hyperparams': {'max_depth': 10, 'n_estimators': 100, 'learning_rate': 0.01}, 'roc': 0.7350013451708366}
Training time: 0.6932089328765869
{'index': 24, 'model_type': 'XGBClassifier', 'hyperparams': {'max_depth': 3, 'n_estimators': 500, 'learning_rate': 0.1}, 'roc': 0.7452246435297283}
Training time: 0.588068962097168
{'index': 25, 'model_type': 'XGBClassifier', 'hyperp

In [20]:
param_grid = [
    # Matern kernel models
    {
        'kernel': Matern(),  
    },
    {
        'kernel': Matern(),  
    },
    # Quadratic (RBF) kernel models  
    {
        'kernel': RBF(),  
    },
    {
        'kernel': RBF(),
    },
    #RationalQuadratic kernel model
    {
        'kernel': RationalQuadratic()
    },
    {
        'kernel': RationalQuadratic()
    }
]

for idx, params in enumerate(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    start_time = time.time()
    model = GaussianProcessClassifier(kernel=params['kernel'])
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    roc = float(roc_auc_score(y_val, y_val_pred))
    training_time = time.time() - start_time
    print(f'Training time: {training_time}')
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'roc': roc 
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)


Training time: 1.687493085861206
{'index': 28, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'roc': 0.698210922787194}
Training time: 1.6234090328216553
{'index': 29, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'roc': 0.7135458703255313}
Training time: 2.3089332580566406
{'index': 30, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'roc': 0.6898036050578424}
Training time: 3.0308525562286377
{'index': 31, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'roc': 0.7209443099273607}
Training time: 3.725635051727295
{'index': 32, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': RationalQuadratic(alpha=1, length_scale=1)}, 'roc': 0.6959913909066452}
Training time: 4.171703815460205
{'index': 33, 'model_type': 'GaussianProcessClassifier', 'hyperparams': {'kernel': RationalQuad

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import Tuple, List, Optional
from copy import deepcopy

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, n_layers=2, layer_size=5):
        super().__init__()
        
        
        layers = []
        
        # Input layer
        layers.append(nn.Linear(input_dim, layer_size))
        layers.append(nn.ReLU())
        
        # Hidden layers (n_layers - 1 additional hidden layers)
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(layer_size, layer_size))
            layers.append(nn.ReLU())

        # Output layer
        layers.append(nn.Linear(layer_size, output_dim))
        # layers.append(nn.Sigmoid())
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)
        

def create_mlp_pytorch(input_dim, output_dim, n_layers=2, layer_size=5, 
                       lr=0.01, l2_reg=0.01):
    """
    Create MLP with specified parameters
    
    Args:
        input_dim: Input dimension
        output_dim: Output dimension
        n_layers: Number of hidden layers
        layer_size: Number of units per hidden layer
        lr: Learning rate
        l2_reg: L2 regularization strength (weight_decay)
    """
    model = MLP(input_dim, output_dim, n_layers, layer_size)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=l2_reg)
    
    return model, optimizer

In [22]:
from torch.utils.data import DataLoader, TensorDataset

In [23]:
hyperparams_list = [
    {'n_layers': 2, 'layer_size': 5, 'lr': 0.01, 'l2_reg': 0.01},
    {'n_layers': 2, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.0005, 'l2_reg': 0.1}
]

for idx, params in enumerate(hyperparams_list):
    
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Create MLP
    model, optimizer = create_mlp_pytorch(
        params['input_dim'],
        params['output_dim'], 
        n_layers=params['n_layers'],
        layer_size=params['layer_size'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])
    
    # Training code would go here
    # model.train() ... etc.
    # Basic EarlyStopping for 1000 epochs
    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.BCEWithLogitsLoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(input=output.squeeze(), target=target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(input=output.squeeze(), target=target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    roc = float(roc_auc_score(all_targets, all_predictions))
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'MLP_PyTorch',
        'hyperparams': params,
        'roc': roc
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 0.5966, Val Loss: 0.5886
Validation loss decreased (inf -> 0.588624). Saving model...
Epoch 2/1000, Train Loss: 0.4605, Val Loss: 0.6357
EarlyStopping counter: 1 out of 20
Epoch 3/1000, Train Loss: 0.3982, Val Loss: 0.6185
EarlyStopping counter: 2 out of 20
Epoch 4/1000, Train Loss: 0.3813, Val Loss: 0.6453
EarlyStopping counter: 3 out of 20
Epoch 5/1000, Train Loss: 0.3806, Val Loss: 0.6991
EarlyStopping counter: 4 out of 20
Epoch 6/1000, Train Loss: 0.3403, Val Loss: 0.6796
EarlyStopping counter: 5 out of 20
Epoch 7/1000, Train Loss: 0.3280, Val Loss: 0.6679
EarlyStopping counter: 6 out of 20
Epoch 8/1000, Train Loss: 0.3152, Val Loss: 0.6786
EarlyStopping counter: 7 out of 20
Epoch 9/1000, Train Loss: 0.3386, Val Loss: 0.7068
EarlyStopping counter: 8 out of 20
Epoch 10/1000, Train Loss: 0.3039, Val Loss: 0.6570
EarlyStopping counter: 9 out of 20
Epoch 11/1000, Train Loss: 0.3452, Val Loss: 0.6037
EarlyStopping counter: 10 out of 20
Epoch 12/1000, Train Loss

In [24]:
def create_mlp_resnet(input_dim, output_dim, block_dim, hidden_dim, num_blocks, lr, l2_reg):
    model = ResNet(
        input_dim = input_dim,
        output_dim = output_dim,
        num_blocks = num_blocks,
        hidden_dim = hidden_dim,
        block_dim = block_dim
    )
    optimizer = optim.Adam(model.parameters(), lr, weight_decay=l2_reg)

    return model, optimizer

In [25]:
specified_configs = [
    # D block D hidden N blocks Learning rate L2 regularisation
    {'block_dim': 16, 'hidden_dim': 8, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.01},
    {'block_dim': 32, 'hidden_dim': 16, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.1},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.01},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.1},
]

for idx, params in enumerate(specified_configs):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    model, optimizer = create_mlp_resnet(
        input_dim = params['input_dim'], 
        output_dim = params['output_dim'],
        num_blocks=params['num_blocks'],
        hidden_dim=params['hidden_dim'],
        block_dim=params['block_dim'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])


    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.BCEWithLogitsLoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output.squeeze(), target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output.squeeze(), target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    roc = float(roc_auc_score(all_targets, all_predictions))
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'ResNet',
        'hyperparams': params,
        'roc': roc
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 0.5132, Val Loss: 0.5996
Validation loss decreased (inf -> 0.599579). Saving model...
Epoch 2/1000, Train Loss: 0.3592, Val Loss: 0.6065
EarlyStopping counter: 1 out of 20
Epoch 3/1000, Train Loss: 0.3690, Val Loss: 0.5763
Validation loss decreased (0.599579 -> 0.576321). Saving model...
Epoch 4/1000, Train Loss: 0.3310, Val Loss: 0.6183
EarlyStopping counter: 1 out of 20
Epoch 5/1000, Train Loss: 0.2822, Val Loss: 0.5865
EarlyStopping counter: 2 out of 20
Epoch 6/1000, Train Loss: 0.2847, Val Loss: 0.7087
EarlyStopping counter: 3 out of 20
Epoch 7/1000, Train Loss: 0.2735, Val Loss: 0.6281
EarlyStopping counter: 4 out of 20
Epoch 8/1000, Train Loss: 0.2707, Val Loss: 0.6155
EarlyStopping counter: 5 out of 20
Epoch 9/1000, Train Loss: 0.2373, Val Loss: 0.7189
EarlyStopping counter: 6 out of 20
Epoch 10/1000, Train Loss: 0.2409, Val Loss: 0.7293
EarlyStopping counter: 7 out of 20
Epoch 11/1000, Train Loss: 0.2788, Val Loss: 0.6455
EarlyStopping counter: 8 out o

In [26]:
models_df = pd.DataFrame(columns=[
    'index', 'model_type', 'hyperparams', 'roc'
])
models_df = pd.DataFrame(models_list)
models_df = models_df.sort_values('roc', ascending=False).reset_index(drop=True)
final_models = models_df.head(10)
final_models

,index,model_type,hyperparams,roc
0,41,ResNet,"{'block_dim': 64, 'hidden_dim': 32, 'num_block...",0.815981
1,35,MLP_PyTorch,"{'n_layers': 2, 'layer_size': 10, 'lr': 0.001,...",0.807237
2,38,ResNet,"{'block_dim': 16, 'hidden_dim': 8, 'num_blocks...",0.805712
3,39,ResNet,"{'block_dim': 32, 'hidden_dim': 16, 'num_block...",0.794503
4,34,MLP_PyTorch,"{'n_layers': 2, 'layer_size': 5, 'lr': 0.01, '...",0.772958
5,40,ResNet,"{'block_dim': 64, 'hidden_dim': 32, 'num_block...",0.768944
6,0,KNeighborsClassifier,{},0.767353
7,22,XGBClassifier,"{'max_depth': 10, 'n_estimators': 100, 'learni...",0.756524
8,2,KNeighborsClassifier,{},0.755448
9,36,MLP_PyTorch,"{'n_layers': 3, 'layer_size': 10, 'lr': 0.001,...",0.749350


In [27]:
def softmax(series):
    exp_x = np.exp(series - series.max())
    return exp_x / exp_x.sum()

In [28]:
final_models['weights'] = softmax(final_models['roc'])
final_models

/tmp/ipykernel_388791/4123806128.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_models['weights'] = softmax(final_models['roc'])


,index,model_type,hyperparams,roc,weights
0,41,ResNet,"{'block_dim': 64, 'hidden_dim': 32, 'num_block...",0.815981,0.103698
1,35,MLP_PyTorch,"{'n_layers': 2, 'layer_size': 10, 'lr': 0.001,...",0.807237,0.102795
2,38,ResNet,"{'block_dim': 16, 'hidden_dim': 8, 'num_blocks...",0.805712,0.102639
3,39,ResNet,"{'block_dim': 32, 'hidden_dim': 16, 'num_block...",0.794503,0.101495
4,34,MLP_PyTorch,"{'n_layers': 2, 'layer_size': 5, 'lr': 0.01, '...",0.772958,0.099331
5,40,ResNet,"{'block_dim': 64, 'hidden_dim': 32, 'num_block...",0.768944,0.098933
6,0,KNeighborsClassifier,{},0.767353,0.098776
7,22,XGBClassifier,"{'max_depth': 10, 'n_estimators': 100, 'learni...",0.756524,0.097712
8,2,KNeighborsClassifier,{},0.755448,0.097607
9,36,MLP_PyTorch,"{'n_layers': 3, 'layer_size': 10, 'lr': 0.001,...",0.749350,0.097014


In [29]:
roc = sum(final_models['roc']*final_models['weights'])
roc

0.7799356352959933